In [43]:
from google_play_scraper import Sort, reviews
import pandas as pd
from datetime import datetime
import time

In [44]:
# Define app IDs (updated based on snippet and web results)
app_ids = {
    "CBE": "com.combanketh.mobilebanking",  # Updated CBE app ID from snippet
    "BOA": "com.boa.apollo",               # Confirmed working (1,822 reviews)
    "Dashen": "com.cr2.amolelight"         # Dashen Mobile Plus
}

In [45]:
# Function to scrape reviews for a single app
def scrape_reviews(app_id, bank_name, count=400):
    try:
        all_reviews = []
        continuation_token = None
        reviews_fetched = 0

        # Fetch reviews in batches until we reach the desired count
        while reviews_fetched < count:
            result, continuation_token = reviews(
                app_id,
                lang='en',
                country='us',  # Consistent with snippet
                sort=Sort.NEWEST,
                count=min(100, count - reviews_fetched),  # Fetch up to 100 reviews per batch
                continuation_token=continuation_token
            )
            all_reviews.extend(result)
            reviews_fetched += len(result)
            if not continuation_token or len(result) == 0:  # No more reviews to fetch
                break
            time.sleep(2)  # Avoid rate limiting within batches

        if len(all_reviews) < count:
            print(f"Warning: Only {len(all_reviews)} reviews collected for {bank_name}, expected {count}")

        data = []
        for review in all_reviews[:count]:  # Limit to requested count
            review_date = review.get('at')
            formatted_date = review_date.strftime('%Y-%m-%d') if isinstance(review_date, datetime) else 'N/A'
            data.append({
                'review': review.get('content', 'N/A'),
                'rating': review.get('score', None),
                'date': formatted_date,
                'bank': bank_name,
                'source': 'Google Play'
            })
        df = pd.DataFrame(data)
        print(f"Collected {len(df)} reviews for {bank_name}")
        return df
    except Exception as e:
        print(f"Error scraping {bank_name}: {str(e)}")
        return pd.DataFrame()

In [46]:
# Scrape reviews for all banks
all_reviews = pd.DataFrame()
for bank, app_id in app_ids.items():
    print(f"Scraping reviews for {bank}...")
    df = scrape_reviews(app_id, bank, count=400)
    all_reviews = pd.concat([all_reviews, df], ignore_index=True)
    time.sleep(5)  # Avoid overwhelming the server

Scraping reviews for CBE...
Collected 400 reviews for CBE
Scraping reviews for BOA...
Collected 400 reviews for BOA
Scraping reviews for Dashen...
Collected 400 reviews for Dashen


In [47]:
# Validate total reviews
total_reviews = len(all_reviews)
print(f"Total reviews scraped: {total_reviews}")
if total_reviews < 1200:
    print(f"Warning: Only {total_reviews} reviews collected, expected at least 1200")

Total reviews scraped: 1200


In [48]:
# Save raw data to CSV
all_reviews.to_csv('raw_bank_reviews.csv', index=False)
print(f"Scraped {total_reviews} reviews and saved to raw_bank_reviews.csv")

Scraped 1200 reviews and saved to raw_bank_reviews.csv


In [49]:
# Check for missing data
missing_percentage = all_reviews.isnull().mean() * 100
print("Missing data percentage:")
print(missing_percentage)

Missing data percentage:
review    0.0
rating    0.0
date      0.0
bank      0.0
source    0.0
dtype: float64
